# Configs — 04: Self-Discovery, Slim Mode, Field-by-Field Docs

**Persona:** Catalog/Collection admin

**Goal:** Demonstrate the standards-aligned self-discovery features added to the composed-config API:

1. **JSON Hyper-Schema `_links`** at the response root — discover supported query parameters and alternate representations without consulting OpenAPI.
2. **`?meta=field`** (default) — lightweight `{field_name: description}` map per class.
3. **`?meta=schema`** — full JSON Schema 2020-12 with `examples` per field (for form-builders).
4. **`?include=scope`** (default) — body shows only configs owned by this scope; upstream-tier configs surface in the hierarchical `inherited` tree (mirrors `configs` shape, `{source: <tier>}` leaves at natural addresses).
5. **`?include=upstream`** — verbose mode (full waterfall in body).
6. **Tier-first tree paths** — driver configs land at `storage.{tier}.drivers` and routing configs at `storage.{tier}.routing` (post-Cycle-D restructure: items/assets/catalog/collection forks group all configs per tier).

The endpoints are unchanged from earlier notebooks; the additions surface as new fields in the response and new query parameters.

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
ADMIN_TOKEN   = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120)

print(f"Base URL    : {BASE_URL}")
print(f"Catalog     : {CATALOG_ID}")
print(f"Collection  : {COLLECTION_ID}")

## Step 1 — Slim default response (`?include=scope`, the new default)

Asking the collection scope returns ONLY configs owned by this collection — `_visibility=collection` configs (routing + driver-tier with sidecars) plus any explicit collection-level overrides. Upstream-tier configs are summarised in the hierarchical `inherited` tree — leaves carry `{source: <tier>}` at the same address the resolved value would land at if inlined. Operators see what exists upstream without 700 rows of code defaults flooding the body.

In [ ]:
resp = client.get(f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
assert resp.status_code == 200, resp.text
body = resp.json()


def _walk_inherited(tree, path=()):
    """Yield (address, source) leaves from the hierarchical inherited tree."""
    if not isinstance(tree, dict):
        return
    if set(tree.keys()) == {"source"}:
        yield path, tree["source"]
        return
    for key, sub in tree.items():
        yield from _walk_inherited(sub, path + (key,))


inherited_tree = body.get("inherited") or {}
inherited_leaves = list(_walk_inherited(inherited_tree))

print("Top-level keys:", sorted(body.keys()))
print(f"\nClasses in body  : {sum(len(v) if isinstance(v, dict) else 0 for v in body['configs'].values())}")
print(f"Inherited leaves : {len(inherited_leaves)}")
print(f"\nSample inherited (address → source):")
for path, src in inherited_leaves[:5]:
    print(f"  {'.'.join(path):60s} ← {src}")

## Step 2 — Self-discovery via the `self` link's `hrefSchema`

Every response carries a response-level `_links` block — post-#517 slimmed to **only** the `self` entry. The `self` link's `hrefSchema` is a JSON Schema 2020-12 doc describing every supported query parameter (`resolved`, `meta`, `include`, `strict`, `links`) with `description` + `examples`. Per-plugin edit affordances now live **inline on each leaf** (see Step 3); the templated response-root `edit` link is gone — every leaf is self-describing.


In [ ]:
for link in body["_links"]:
    print(f"{link['rel']:12s} {link['method']:6s} {link['href']}")
    if link.get('title'):
        print(f"             ↳ {link['title']}")

# The self link's hrefSchema lists every supported query param incl. `links`
self_link = next(l for l in body["_links"] if l["rel"] == "self")
print("\n=== Query parameters discoverable via hrefSchema ===")
for name, spec in self_link["hrefSchema"]["properties"].items():
    examples = spec.get("examples", [])
    print(f"  ?{name}={spec.get('default')!r:12} (examples: {examples})")
    print(f"     {spec['description']}\n")


## Step 3 — Field-by-field documentation lives **inline** on each leaf (`?meta=field`)

Post-#517 the parallel top-level `meta` tree is gone — single source of truth. With `?meta=field` (the default), every in-scope plugin leaf carries a `_meta.docs` sibling: `{field_name: description}`. The leaf is fully self-describing — no cross-walk needed.


In [ ]:
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"meta": "field"},
)
body = resp.json()


def _find_leaf(tree, class_key):
    """Walk the `configs` tree to find a plugin leaf by class_key."""
    stack = [tree or {}]
    while stack:
        node = stack.pop()
        if not isinstance(node, dict):
            continue
        if class_key in node and isinstance(node[class_key], dict) and (
            "_meta" in node[class_key] or "_links" in node[class_key]
            or any(k for k in node[class_key] if not k.startswith("_"))
        ):
            return node[class_key]
        for v in node.values():
            stack.append(v)
    return None


# Items driver leaf — field docs inline under `_meta.docs`
items_leaf = _find_leaf(body.get("configs") or {}, "items_postgresql_driver")
if items_leaf and items_leaf.get("_meta"):
    print("=== ItemsPostgresqlDriverConfig — field-level docs (inline) ===")
    for field_name, desc in (items_leaf["_meta"].get("docs") or {}).items():
        print(f"\n  {field_name}:\n    {desc[:140]}{'...' if len(desc) > 140 else ''}")


## Step 4 — Full JSON Schema with `examples` (`?meta=schema`, for form-builders)

With `?meta=schema`, the same `_meta` sibling carries `json_schema` instead of `docs` — full JSON Schema 2020-12 per class. Each `Field(...)`'s `description` AND `examples` come through.


In [ ]:
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"meta": "schema"},
)
body = resp.json()

wp_leaf = _find_leaf(body.get("configs") or {}, "items_write_policy")
if wp_leaf and wp_leaf.get("_meta", {}).get("json_schema"):
    schema = wp_leaf["_meta"]["json_schema"]
    print("=== ItemsWritePolicy — schema with examples (inline) ===\n")
    for field_name, spec in schema.get("properties", {}).items():
        examples = spec.get("examples")
        if examples is not None:
            print(f"  {field_name:30s} examples = {examples}")


## Step 5 — Per-plugin edit affordances via `?links=full`

This is the headline addition from PR #518. With `?links=minimal` (the new default) or `?links=full`, every in-scope plugin leaf carries a `_links` array with four contextual rels:

| `rel`         | method  | what it does |
|---------------|---------|--------------|
| `self`        | GET     | Read this plugin at this scope. |
| `edit`        | PUT     | Replace the plugin config at this scope. |
| `edit`        | DELETE  | Clear the scope override (fall back to parent tier). |
| `describedby` | GET     | JSON Schema for this plugin class (scope-agnostic). |

`full` adds `title` strings that name the class + tier so each link is copy-pasteable straight from the response. Opt out with `?links=none` for a terse payload.


In [ ]:
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"links": "full", "meta": "field"},
)
body = resp.json()

items_leaf = _find_leaf(body.get("configs") or {}, "items_postgresql_driver")
if items_leaf and items_leaf.get("_links"):
    print("=== items_postgresql_driver — per-leaf affordances (links=full) ===\n")
    for link in items_leaf["_links"]:
        print(f"  {link['rel']:12s} {link['method']:6s} {link['href']}")
        if link.get('title'):
            print(f"               ↳ {link['title']}")
else:
    print("No items_postgresql_driver leaf at this scope; try a different class_key.")


## Step 6 — Follow a real `edit` (PUT) round-trip and a `describedby` link

The `edit` PUT is a **replace**: send the current body back to confirm round-trip works, then hit `describedby` to fetch the schema operators would feed to a form-builder. This replaces the old templated-edit walkthrough — every URL is concrete now.


In [ ]:
if items_leaf and items_leaf.get("_links"):
    # Strip protocol-level keys before round-tripping
    payload = {k: v for k, v in items_leaf.items() if not k.startswith("_")}

    edit_put = next(
        (l for l in items_leaf["_links"] if l["rel"] == "edit" and l["method"] == "PUT"),
        None,
    )
    described = next(
        (l for l in items_leaf["_links"] if l["rel"] == "describedby"),
        None,
    )

    if edit_put:
        print(f"PUT {edit_put['href']}")
        # Live round-trip:
        # r = client.put(edit_put['href'], json=payload)
        # print(f"  → {r.status_code}")
        print(f"  (would replace with current body; {len(payload)} top-level fields)")

    if described:
        r = client.get(described['href'])
        if r.status_code == 200:
            schema = r.json()
            print(f"\nGET {described['href']} → {r.status_code}")
            print(f"  title  : {schema.get('title')}")
            print(f"  fields : {list((schema.get('properties') or {}).keys())[:8]}")


## Step 5 — Driver-tier sidecars grouped by entity

PostgreSQL drivers carry typed sidecar tables that compose each row.  The four valid items-tier sidecar types (post-PR-G2 split) are:

| `sidecar_type`     | Owns physical table              | Holds                                                                 |
|--------------------|----------------------------------|-----------------------------------------------------------------------|
| `geometries`       | `{items}_geometries`             | `geom`, `bbox`, `geohash` (STORED GENERATED)                          |
| `attributes`       | `{items}_attributes`             | `external_id`, JSONB / columnar attributes                            |
| `item_metadata`    | `{items}_item_metadata`          | `title`, `description`, `keywords` (multilanguage)                    |
| `stac_metadata`    | `{items}_stac_metadata`          | `external_extensions`, `external_assets`, `extra_fields` (JSONB)      |

Collection and catalog tiers have parallel sidecar splits (`collection_core`/`collection_stac`, `catalog_core`/`catalog_stac`).  An OGC-only deployment that doesn't mount the STAC extension carries no `_stac_metadata` companion table.

Driver-tier configs land at `configs.platform.catalog.{tier}.drivers.{driver_ref}` (post-Cycle-D tier-first paths) — the tier segment encodes the entity bucket directly.

## Step 7 — Driver-tier sidecars grouped by entity


In [ ]:
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"meta": "field"},
)
body = resp.json()

# Walk driver-tier configs grouped by entity (tier-first: storage.{tier}.drivers).
storage_tree = body["configs"].get("storage", {})
print("=== Driver-tier configs grouped by entity (tier-first paths) ===\n")
for entity_bucket in ("items", "assets", "catalog", "collection"):
    drivers = storage_tree.get(entity_bucket, {}).get("drivers", {})
    if not drivers:
        continue
    print(f"--- entity: {entity_bucket} ---")
    for class_key, payload in drivers.items():
        sidecars = payload.get("sidecars", []) if isinstance(payload, dict) else []
        print(f"  {class_key}")
        if sidecars:
            for sc in sidecars:
                print(f"    + sidecar: {sc.get('sidecar_type', sc)} (enabled={sc.get('enabled', '?')})")
    print()

## Step 8 — Asset routing at catalog scope + following its `edit` link

Asset routing config lives at catalog scope. With `?links=full`, the `items_routing` / `asset_routing_config` leaf carries its own 4-rel `_links` set — the parent routing leaf points at the routing-config plugin CRUD (orthogonal to the per-entry `driver-config` links emitted by `_build_routing_refs`).


In [ ]:
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}",
    params={"meta": "field", "links": "full"},
)
body = resp.json()

asset_routing = (
    body["configs"]
    .get("storage", {})
    .get("assets", {})
    .get("routing", {})
    .get("asset_routing_config")
)
if asset_routing:
    print("=== AssetRoutingConfig — operations dispatch ===")
    for op, entries in (asset_routing.get("operations") or {}).items():
        for e in entries:
            print(f"  {op:8s} → {e.get('driver_ref'):30s} on_failure={e.get('on_failure')}")
            # Per-entry driver-config link is still emitted by _build_routing_refs
            for link in (e.get("_links") or []):
                print(f"           · {link['rel']:14s} {link['method']:6s} {link['href']}")

    print("\n=== Parent leaf affordances (links=full) ===")
    for link in (asset_routing.get("_links") or []):
        print(f"  {link['rel']:12s} {link['method']:6s} {link['href']}")
        if link.get('title'):
            print(f"               ↳ {link['title']}")


## Step 9 — Opt-in to verbose mode (`?include=upstream`)

Existing scripts that consume the full waterfall opt back in with `?include=upstream`.


In [ ]:
slim = client.get(f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}").text
full = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"include": "upstream"},
).text

print(f"Slim mode (default)   : {len(slim):>7,d} bytes")
print(f"Upstream (verbose)    : {len(full):>7,d} bytes")
print(f"Reduction             : {(1 - len(slim)/len(full))*100:>6.1f}% smaller")

## Summary

- Response-level `_links` is slimmed to a single `self` entry; its `hrefSchema` advertises all query params (`resolved`, `meta`, `include`, `strict`, `links`) with `description` + `examples`.
- `?meta=field` (default) and `?meta=schema` attach a `_meta` sibling **inline on each in-scope plugin leaf** (`{docs: …}` or `{json_schema: …}`). The parallel top-level `meta` tree is retired.
- `?links=minimal` (default) and `?links=full` attach a `_links` sibling on each in-scope plugin leaf with four rels: `self` GET, `edit` PUT (replace), `edit` DELETE (clear override), `describedby` GET. `full` carries contextual titles naming class + tier; opt out with `?links=none`.
- Multi-instance refs use `ref_key` for `self`/`edit`; `describedby` always points at the canonical `class_key` (schema is per-class).
- Inherited-tree placeholders carry NEITHER `_meta` NOR `_links` — they're not actionable at the active scope.
- `?include=scope` (default) slim body + hierarchical `inherited` tree with `{source: <tier>}` leaves; `?include=upstream` is the verbose alternate.
- Tier-first paths: driver configs at `configs.platform.catalog.{tier}.drivers.{driver_ref}`, routing at `configs.platform.catalog.{tier}.routing.{class_key}`.
- Items-tier sidecars (`geometries`, `attributes`, `item_metadata`, `stac_metadata`) dispatched via `SidecarConfigRegistry`.


In [ ]:
# Final: list sidecars + show the real (not templated) edit URL for the driver leaf
resp = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"links": "full"},
)
body = resp.json()

items_pg = (
    body["configs"]
    .get("storage", {})
    .get("items", {})
    .get("drivers", {})
    .get("items_postgresql_driver", {})
)
sidecars = items_pg.get("sidecars") or []

print("=== Items-tier sidecars on this collection ===\n")
for sc in sidecars:
    sc_type = sc.get("sidecar_type", "?")
    extra = {k: v for k, v in sc.items() if k != "sidecar_type"}
    print(f"  + {sc_type}{f' ({extra})' if extra else ''}")

# Real edit URL — no template substitution needed
edit_put = next(
    (l for l in (items_pg.get("_links") or [])
     if l["rel"] == "edit" and l["method"] == "PUT"),
    None,
)
if edit_put:
    print(f"\n=== Real edit URL (no template) ===\n  PUT {edit_put['href']}")
    print("\n=== Expected 422 on unknown sidecar_type (illustrative payload) ===")
    print('  body  = {"sidecars": [{"sidecar_type": "geohash"}]}')
    print("  error = sidecar_type 'geohash' not registered;")
    print("          registered types: ['attributes', 'geometries', 'item_metadata', 'stac_metadata']")
